In [ ]:
import time

WORKER_ID = 1  # change only this value to 1..5 for the five notebooks
MAX_DOCS = 1000  # set to None for a full/resume run
MAX_RUNTIME_HOURS = 10.5
CHECKPOINT_EVERY_DOCS = 5000
GLOBAL_CONCURRENCY = 32
PER_DOMAIN_CONCURRENCY = 2

STARTED_EPOCH = time.time()
INPUT_DATASET_SLUG = "vibiomir-crawl-input"
CODE_DATASET_SLUG = "vibiomir-crawler-code"
APP_DIR = "/kaggle/working/app"
OUTPUT_DIR = f"/kaggle/working/crawl_worker_{WORKER_ID}"


In [ ]:
import subprocess
import shutil
import sys
import zipfile
from pathlib import Path

if WORKER_ID not in range(1, 6):
    raise ValueError("WORKER_ID must be 1, 2, 3, 4, or 5")
if MAX_DOCS is not None and (not isinstance(MAX_DOCS, int) or MAX_DOCS <= 0):
    raise ValueError("MAX_DOCS must be a positive integer or None")
if not ((3, 12) <= sys.version_info[:2] < (3, 14)):
    raise RuntimeError("This crawler requires Python 3.12 or 3.13")

kaggle_input = Path('/kaggle/input')
worker_filename = f'worker_{WORKER_ID}_kaggle.parquet'
input_candidates = sorted(
    path for path in kaggle_input.rglob(worker_filename)
    if INPUT_DATASET_SLUG in path.parts
)
if len(input_candidates) != 1:
    raise RuntimeError(
        f'Expected exactly one {worker_filename} under Dataset '
        f'{INPUT_DATASET_SLUG!r}; found: {[str(path) for path in input_candidates]}'
    )
INPUT_FILE = str(input_candidates[0])
assignment_file = input_candidates[0].parent / 'domain_assignment.parquet'
if not assignment_file.is_file():
    raise FileNotFoundError(assignment_file)

bundle_candidates = sorted(
    path for path in kaggle_input.rglob('crawler_bundle.zip')
    if CODE_DATASET_SLUG in path.parts
)
extracted_roots = sorted({
    path.parent.parent
    for path in kaggle_input.rglob('crawler/kaggle_runner.py')
    if CODE_DATASET_SLUG in path.parts
})
if len(bundle_candidates) > 1 or (not bundle_candidates and len(extracted_roots) != 1):
    raise RuntimeError(
        'Could not identify one crawler code input; '
        f'archives={[str(path) for path in bundle_candidates]}, '
        f'extracted_roots={[str(path) for path in extracted_roots]}'
    )

app_dir = Path(APP_DIR)
if app_dir.exists():
    shutil.rmtree(app_dir)
if bundle_candidates:
    app_dir.mkdir(parents=True)
    with zipfile.ZipFile(bundle_candidates[0]) as archive:
        for name in archive.namelist():
            member = Path(name)
            if member.is_absolute() or '..' in member.parts:
                raise RuntimeError(f'Unsafe bundle member: {name}')
        archive.extractall(app_dir)
else:
    shutil.copytree(extracted_roots[0], app_dir)
runner_file = app_dir / 'crawler' / 'kaggle_runner.py'
if '--target-limit' not in runner_file.read_text(encoding='utf-8'):
    raise RuntimeError('Crawler code Dataset is outdated; upload the latest bundle version')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-input', '-r',
     str(app_dir / 'requirements-kaggle.txt')],
    check=True,
)
sys.path.insert(0, str(app_dir))
import crawler  # noqa: E402
print(f"Using crawler {crawler.__version__} from {app_dir}")


In [ ]:
command = [
    sys.executable, '-m', 'crawler.kaggle_runner',
    '--worker-id', str(WORKER_ID),
    '--input-file', INPUT_FILE,
    '--output-dir', OUTPUT_DIR,
    '--config', str(app_dir / 'config' / 'crawler.yaml'),
    '--resume-root', '/kaggle/input',
    '--max-runtime-hours', str(MAX_RUNTIME_HOURS),
    '--checkpoint-every-docs', str(CHECKPOINT_EVERY_DOCS),
    '--global-concurrency', str(GLOBAL_CONCURRENCY),
    '--per-domain-concurrency', str(PER_DOMAIN_CONCURRENCY),
    '--deadline-epoch', str(STARTED_EPOCH + MAX_RUNTIME_HOURS * 3600),
]
if MAX_DOCS is not None:
    command.extend(['--target-limit', str(MAX_DOCS)])
subprocess.run(command, cwd=app_dir, check=True)
print(f"Worker output: {OUTPUT_DIR}")
